### Week 7: Outlier Detection and Data Quality

Note: Extreme values in price, square footage, or days on market can distort market averages and trends. 


In [41]:
import pandas as pd
import numpy as np

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.width', 1000)


In [42]:
sold = pd.read_parquet('data/sold_key_metrics.parquet')
listing = pd.read_parquet('data/listing_key_metrics.parquet')


In [ ]:


key_columns = ['ClosePrice', 'LivingArea', 'DaysOnMarket']
new_col_names = ['close_price_outliers', 'living_area_outliers', 'days_on_market_outliers']

for i in range(len(key_columns)):
    key = key_columns[i]
    col = new_col_names[i]

    Q1 = sold[key].quantile(0.25)
    Q3 = sold[key].quantile(0.75)
    IQR = Q3-Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    sold[col] = (sold[key] < lower) | (sold[key] > upper)
    #print(sold[col])





In [44]:
#invalid prices is when the closing price is less than or equal to 0

sold['invalid_close_price'] = sold['ClosePrice'] <= 0
sold.to_parquet('data/sold_price_flagged.parquet', index = False)

In [45]:
#creating a sepearate filtered analysis dataset

sold_separate = sold[
    (sold['close_price_outliers'] == False) & 
    (sold['living_area_outliers'] == False) & 
    (sold['days_on_market_outliers'] == False) & 
    (sold['invalid_close_price'] == False)
].copy()

sold_separate.to_parquet('data/sold_non_outliers.parquet', index = False)

In [ ]:
#Before filter

print("Before filter:")
for key in key_columns: 
    print(f"Median {key}: {sold[key].median():.2f}")

#after filter
print("\nAfter filter")
for key in key_columns:
    print(f"Median {key}: {sold_separate[key].median():.2f}")


print(f"\n\nNumber of Rows Before: {len(sold)}")
print(f"Number of Rows After: {len(sold_separate)}")
print(f"\nRecords removed: {len(sold) - len(sold_separate)}")



Before filter:
Median ClosePrice: 825000.00
Median LivingArea: 1642.00
Median DaysOnMarket: 19.00

After filter
Median ClosePrice: 786000.00
Median LivingArea: 1569.00
Median DaysOnMarket: 17.00


Number of Rows Before: 262310
Number of Rows After: 222352

Records removed: 39958



Written Comparison: 


In this notebook, I used the Interquartile Range method to identify outliers to account for outliers in the key columns. Afterwards, I ensured that no rows were removed from the original dataset. Before the rows were filtered, there were 262,310 rows in the dataset. After the rows were filtered, were 222,352 rows are in the dataset. The median values all decrease after the the datasets were filtered. 


